再往下看一层 SGLang 具体的初始化：

1. 调用父类构造函数并设置配置和设备网格。
2. 通过 _initialize_tools() 初始化工具 schemas、map 和解析器，支持 Multi-turn 对话中的工具使用。
3. 初始化 SGLang 推理所需的分布式环境。
4. 通过 _verify_config() 验证模型配置。
5. 通过 _init_inference_engine() 初始化 SGLang 推理引擎。
6. 通过 _init_sampling_params() 初始化生成序列的采样参数。
7. 设置 Tokenizer 和 padding token ID。

## SGLangRollout.init 部分源码

## SGLangRollout._initialize_tools() 函数用于初始化 Multi-turn 对话中的工具。

1. 如果没有工具配置路径，则返回空列表和字典。
2. 从配置文件加载工具并初始化工具列表。
3. 创建 OpenAI 格式的工具 schema 和工具名称到工具对象的映射。
4. 根据 Tokenizer 类型确定工具调用解析器。
5. 为 SGLang 创建 Tool 对象。
6. 实例化 FunctionCallParser。

## SGLangRollout.AsyncEngine

关于 SGLangRollout 调用 tool 的部分，我们在下文的训练循环中再展开，这里先讨论完 SGLang 的初始化。为了调用 SGLang engine 的接口，verl 进行了一层封装，实现了我们对 SGLang 除开 rollout 之外的所有接口：

1. release and resume memory occupation：在训练时释放掉显存占用并在训练后恢复。
2. update weights from tensor：训练结束后更新模型权重。
3. flush cache：模型参数更新后刷新 KV cache，因为之前的 KV cache 已经失效了。

这里涉及到了非常深入的内存管理问题，读者对 SGLang engine 在 verl 里的显存管理感兴趣，欢迎阅读标哥的博客 optimizing Memory Usage in verl，写的非常深入浅出。

### SGLangRollout.AsyncEngine 源码

## SGLangRollout._init_inference_engine() 初始化了封装的 AsyncEngine